In [ ]:
import util.data_loading
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load in Data

In [ ]:
debt_df= util.data_loading.load_raw_data('debt_2003_2025 _clean.csv')
debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')

debt_df2= util.data_loading.load_raw_data('debt_pre_2003.csv')
debt_df2 = debt_df2.T

debt_df2.columns = debt_df2.iloc[0]
debt_df2 = debt_df2[1:]

debt_df2 = debt_df2.reset_index()
debt_df2 = debt_df2.rename(columns={"index": "quarter"})

debt_df = util.data_loading.clean_dates(debt_df)
debt_df2 = util.data_loading.clean_dates(debt_df2)

debt_all = pd.concat(
    [debt_df2, debt_df],
    axis=0,          # stack rows
    ignore_index=True
)

units_all =  util.data_loading.load_raw_data('housing_units_all.csv')

median = util.data_loading.load_raw_data('MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv')
median_all  = util.data_loading.clean_median(median)

population = util.data_loading.load_raw_data('population_clean.csv')



In [ ]:
# median_all = median_all[median_all["Year"] >= 2000 & median_all["Year"] <= 2024]
median_all

In [ ]:

units_all

In [ ]:
debt_all

In [ ]:
mortgage_debt =  util.data_loading.get_debt(debt_all,"Mortgage")
mortgage_debt = mortgage_debt[(mortgage_debt["Year"] >= 2000) & (mortgage_debt["Year"] <= 2024)]
mortgage_debt

In [ ]:
mortgage_debt = mortgage_debt.apply(pd.to_numeric, errors="coerce")

In [ ]:
#TODO: update so that units can be extraced based on just county name
ca_units = util.data_loading.get_unit_estimates(units_all, "California")
ca_units = ca_units[(ca_units["Year"] >= 2000) & (ca_units["Year"] <= 2024)]
ca_units

## Function for running Regression on specific cities wihtout having to run each cell

In [ ]:
def regression(city, debt_type = "Mortgage"):

    # Median Price specific area
    median = util.data_loading.get_median_prices(median_all,city)
    # median = median[(median["Year"] >= 1999) & (median["Year"] <= 2024)]
    # median["median_pct_change"] = median[city].pct_change()

    # Unit Estimates specific area
    units = util.data_loading.get_unit_estimates(units_all, city + " County")
    units = units[(units["Year"] >= 1999) & (units["Year"] <= 2024)]
    units["units_pct_change"] = units["units"].pct_change()

    #Debt
    debt =  util.data_loading.get_debt(debt_all,debt_type)
    debt = debt[(debt["Year"] >= 1999) & (debt["Year"] <= 2024)]
    debt = debt.apply(pd.to_numeric, errors="coerce")
    debt[debt_type+"_pct_change"] = debt[debt_type].pct_change()


    # Population national
    population = util.data_loading.load_raw_data('population_clean.csv')
    population["pop_pct_change"] = population["population"].pct_change()

    features = (
    median.merge(debt, on="Year", how="inner")
       .merge(units, on="Year", how="inner")
    .merge(population, on="Year", how="inner")
    )

    features = features.fillna(0)
    # print("Data Types: \n", features.dtypes)
    # print(features)

    X = features[["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]]

    y = features[city]
    #
    X = sm.add_constant(X[["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]])


    model = sm.OLS(y, X).fit()
    print(model.summary())


In [ ]:
regression("Los Angeles")

In [ ]:
regression("Orange")


In [ ]:
regression("Riverside")


In [ ]:
regression("San Bernardino")


In [ ]:
regression("San Diego")


In [ ]:
regression("Ventura")


## Final Regression Model Custom indepedent variables for each city

In [ ]:
def regression2(city, debt_type = "Mortgage", f=None):

    # Median Price specific area
    median = util.data_loading.get_median_prices(median_all,city)
    # median = median[(median["Year"] >= 1999) & (median["Year"] <= 2024)]
    # median["median_pct_change"] = median[city].pct_change()

    # Unit Estimates specific area
    units = util.data_loading.get_unit_estimates(units_all, city + " County")
    units = units[(units["Year"] >= 1999) & (units["Year"] <= 2024)]
    units["units_pct_change"] = units["units"].pct_change()

    #Debt
    debt =  util.data_loading.get_debt(debt_all,debt_type)
    debt = debt[(debt["Year"] >= 1999) & (debt["Year"] <= 2024)]
    debt = debt.apply(pd.to_numeric, errors="coerce")
    debt[debt_type+"_pct_change"] = debt[debt_type].pct_change()


    # Population national
    population = util.data_loading.load_raw_data('population_clean.csv')
    population["pop_pct_change"] = population["population"].pct_change()

    features = (
    median.merge(debt, on="Year", how="inner")
       .merge(units, on="Year", how="inner")
    .merge(population, on="Year", how="inner")
    )
    features = features.fillna(0)
    # print("Data Types: \n", features.dtypes)
    # print(features)

    X = features[f]

    y = features[city]
    #
    X = sm.add_constant(X[f])


    model = sm.OLS(y, X).fit()
    print(model.summary())


In [ ]:
regression2("Los Angeles",f = [ "units", "units_pct_change",  "Mortgage_pct_change"])

In [ ]:
regression2("Orange",f = [ "units", "population", "Mortgage_pct_change"])

In [ ]:
regression2("Riverside",f = ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change"])

In [ ]:
regression2("San Bernardino",f = ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change"])


In [ ]:
regression2("San Diego", f = ["Mortgage", "units", "population", "Mortgage_pct_change"])


In [ ]:
regression2("Ventura", f = [ "Mortgage","population","units_pct_change", "Mortgage_pct_change"])
# ["Mortgage", "units", "population", "units_pct_change", "pop_pct_change", debt_type+"_pct_change"]

In [ ]:
regression2("Imperial", f = ["Mortgage", "units", "population", "pop_pct_change", "Mortgage_pct_change"])
